In [1]:
import re
import pandas as pd

mgf_path = "/rd1/user/lit/project/sORFs/src/C8_processed/CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020_uncalibrated.mgf"
psm_path = "/rd1/user/lit/project/sORFs/analysis/20250523_human_ms_run/output/db_search_20250523/Trypsin/CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2_3_1_7020_d/psm.tsv"
output_path = "./SPEQ/test_20250705/label.csv"

def spectrum_to_title(spectrum):
    # 去掉补零
    parts = spectrum.split('.')
    # 只保留数字部分去掉前导零
    parts = [str(int(p)) for p in parts[1:]]  # 跳过第一个（文件名部分）
    # 获取SCANS号（最后一位）
    scan_num = parts[-2]  # 倒数第二个是scan号
    # 拼接
    base = spectrum.split('.')[0]
    title = f"{base}.{'.'.join(parts)}_SCANS{scan_num}"
    return title

# 1. 读取psm.tsv，获取所有Spectrum对应的title
psm_df = pd.read_csv(psm_path, sep='\t')
psm_titles = set(psm_df['Spectrum'].apply(spectrum_to_title))

# 2. 读取mgf，提取所有TITLE
titles = []
with open(mgf_path) as f:
    for line in f:
        if line.startswith('TITLE='):
            title = line.strip().split('=', 1)[1]
            titles.append(title)

# 3. 生成label
rows = []
for title in titles:
    label = 1 if title in psm_titles else 0
    rows.append({'Title': title, 'Label': label})

# 4. 写出label.tsv
label_df = pd.DataFrame(rows)
label_df.to_csv(output_path, sep=',', index=False)

In [4]:
output_path = "./SPEQ/test_20250705/labels.csv"
label_df.to_csv(output_path, sep=',', index=False)